# Does the answer depend on my choices?

*Question, Intuition, Math, Code, Assumptions, How it breaks*

Four numbers in a config file decide who the best footballer of the last
twenty-five years is.

```
gate_percentile = 40      the floor every requirement must clear
min_minutes     = 900     how much football a season needs to count
min_seasons     = 3       how many seasons a career needs to be ranked
```

There used to be a fourth, `prior_minutes = 900`, and finding out what it did is
how this chapter earned its place. More on that at the end.

I picked all of these. Not one is derived from anything, and until this chapter
nothing in the project defended any of them, which is the most obvious hole in a
book that spends thirteen chapters demanding evidence for everything else.

So this chapter attacks them.

## 1. Question

If I had picked different constants, would I have got a different answer?

## 2. Intuition

There is a specific failure this is looking for. A result that moves when you
jiggle a threshold is not a result about football, it is a result about the
threshold. The gate is at the 40th percentile; if the winner changes at the 45th,
then "Messi is the best footballer" really means "Messi is the best footballer
when you draw the line exactly here", and nobody would accept that phrased
honestly.

The test is mechanical. Re-run the whole thing at every plausible setting and see
what survives. There is no cleverness in it, which is why there is no excuse for
not having done it.

## 3. Math

Two things get measured for each setting, and they answer different questions.

**Who wins**, and how much of the top ten survives. This is the question a reader
cares about.

**Spearman's $\rho$ against the shipped ranking**, computed on the players
present in both. Ranks rather than scores, because the scores are re-standardised
each run and are not comparable across settings.

That second measure has a trap in it worth naming. It is computed on the
*intersection*, so a setting that throws half the population out can still score
$\rho \approx 1$ on whoever is left. It says "the survivors are ordered the same
way", not "nothing changed". The qualifier counts are printed next to it for
exactly that reason.

## 4. Code

Everything below re-derives the ranking from the committed sample, using the
same `_rank_group` the pipeline calls. First, proof that this reproduces what the
book actually publishes, because a sensitivity analysis run on different
machinery would be worthless.

In [1]:
import dataclasses

import pandas as pd

from gambeta import kit, needs
from gambeta.cli import _rank_group

cfg0 = kit.load()
seasons = pd.read_parquet("../data/sample/player_season_scored.parquet")
published = pd.read_parquet("../data/sample/ranking.parquet")


def rank_with(cfg: kit.Config) -> pd.DataFrame:
    """Re-derive the outfield ranking under one configuration."""
    return _rank_group(seasons[seasons["minutes"] >= cfg.min_minutes], needs.OUTFIELD, cfg)[0]


baseline = rank_with(cfg0)
mine = list(baseline[baseline["qualified"]]["player"])
theirs = list(published[published["qualified"]]["player"])
same = mine == theirs
print(f"reproduces the published ranking exactly: {same}")
print(f"{len(baseline):,} ranked, {int(baseline['qualified'].sum())} qualified")

reproduces the published ranking exactly: True
5,508 ranked, 343 qualified


Now the sweep. Every setting I would defend in an argument, and a few I would
not.

In [2]:
def describe(parameter: str, value: object, ranking: pd.DataFrame) -> dict:
    q, qb = ranking[ranking["qualified"]], baseline[baseline["qualified"]]
    kept = len(set(q.head(10)["player"]) & set(qb.head(10)["player"]))
    a = q.reset_index(drop=True).reset_index().set_index("player")["index"]
    b = qb.reset_index(drop=True).reset_index().set_index("player")["index"]
    both = a.index.intersection(b.index)
    # Spearman without scipy is Pearson on the ranks.
    rho = a[both].rank().corr(b[both].rank()) if len(both) > 2 else float("nan")
    return {
        "parameter": parameter,
        "value": value,
        "ranked": len(ranking),
        "qualified": len(q),
        "winner": q.iloc[0]["player"],
        "top 10 kept": f"{kept}/10",
        "rho": round(rho, 4),
    }


SWEEP = {
    "gate_percentile": (20, 30, 40, 50, 60),
    "min_seasons": (2, 3, 5, 8),
    "min_minutes": (900, 1350, 1800),
}

rows = [describe("as shipped", "", baseline)]
for parameter, values in SWEEP.items():
    for value in values:
        tuned = dataclasses.replace(cfg0, **{parameter: value})
        rows.append(describe(parameter, value, rank_with(tuned)))

pd.DataFrame(rows).style.format({"rho": "{:.4f}"}).hide(axis="index")

parameter,value,ranked,qualified,winner,top 10 kept,rho
as shipped,,5508,343,Lionel Messi,10/10,1.0000
gate_percentile,20,5508,1827,Lionel Messi,10/10,0.9848
gate_percentile,30,5508,853,Lionel Messi,10/10,0.9927
gate_percentile,40,5508,343,Lionel Messi,10/10,1.0000
gate_percentile,50,5508,136,Lionel Messi,7/10,1.0000
gate_percentile,60,5508,38,Lionel Messi,7/10,1.0000
min_seasons,2,7218,516,Lionel Messi,10/10,0.9995
min_seasons,3,5508,343,Lionel Messi,10/10,1.0000
min_seasons,5,3356,233,Lionel Messi,10/10,0.9992
min_seasons,8,1510,87,Lionel Messi,7/10,0.9964


Read the `winner` column first. **Lionel Messi is first under every single
setting**, including the ones I would argue against: a gate so tight that
thirty-eight players survive, and a career floor so high that most of the modern
game is excluded.

That is the result this chapter exists to produce. The constants decide **how
many** players qualify, which swings from 1,829 to 38. They do not decide who
wins.

### Where it does move

The top ten holds completely across the settings I would defend, and loses three
places at the extremes. It is worth knowing which three, because they are the
same three every time.

In [3]:
top10 = list(baseline[baseline["qualified"]].head(10)["player"])

for parameter, value in [("gate_percentile", 50), ("min_seasons", 8), ("min_minutes", 1800)]:
    r = rank_with(dataclasses.replace(cfg0, **{parameter: value}))
    gone = [p for p in top10 if p not in list(r[r["qualified"]].head(10)["player"])]
    print(f"{parameter} = {value}")
    for player in gone:
        row = r[r["player"] == player]
        reason = row["failed"].item() if len(row) else "not ranked at all"
        print(f"    {player:20s} {reason}")
    print()

gate_percentile = 50
    Kylian Mbappé        reliability
    Robert Lewandowski   discipline
    Karim Benzema        reliability



min_seasons = 8
    Erling Haaland       not ranked at all
    Kylian Mbappé        reliability
    Karim Benzema        reliability



min_minutes = 1800
    Kylian Mbappé        availability, reliability
    Robert Lewandowski   discipline
    Karim Benzema        reliability



One of those constants did nothing at all, and this sweep is how it was found.

In [4]:
import inspect

from gambeta import level
from gambeta.cli import _rank_group as pipeline_stage

body = inspect.getsource(pipeline_stage)
print(f"does the ranking pipeline call shrink()?   {'shrink' in body}")
print(f"does gambeta.level still define shrink()?  {hasattr(level, 'shrink')}")
print(f"is prior_minutes still a config field?     {'prior_minutes' in cfg0.__dataclass_fields__}")

does the ranking pipeline call shrink()?   False
does gambeta.level still define shrink()?  False
is prior_minutes still a config field?     False


When this chapter first ran, there was a fourth parameter in that table.
Setting `prior_minutes` to **zero** and to **ten thousand** produced
byte-identical rankings, because nothing in the pipeline ever read it.
`level.shrink` existed, was correct, was tested, and its only caller was its own
unit test. The normalisation chapter taught it as though the ranking used it.

The cells above are what is left after acting on that. The parameter and the
function are gone, and the era chapter now describes minutes-weighted pooling,
which is what the code has always actually done.

The gap between the two is real and worth understanding rather than papering
over. Careers are pooled with each season weighted by its minutes, so a short
season already counts for less. What shrinkage would have added is the other
half: pulling a short season's *score* toward the mean, so that it is not merely
quieter but also less extreme. This project does not do that. It uses a hard
900-minute floor instead, which is the blunt version of the same idea, and
`consistency` still takes an unweighted percentile where a short season gets a
full vote.

Deleting it rather than wiring it in was the choice, on the grounds that
minutes-weighted pooling is already a correction in the same direction and
doing both risks correcting twice. The honest closing of the gap is not a better
constant, it is the hierarchical model in `PENDING.md`, which would estimate how
far to trust each season from the data.

**What this chapter can claim is narrow and worth stating plainly.** Nothing
above proves the constants are well chosen. It shows the answer does not depend
on them, which is a different and much weaker thing, and it is the only one of
the two that a sweep can establish.